# LLM-as-a-Judge for customer-support replies

A local LLM (Qwen2.5-7B-Instruct via Ollama) answers three yes/no quality questions about 50 support replies.
I compare its answers with my own labels. See `README.md` for the findings.

**Run:** Kernel → Restart & Run All. With `RUN_JUDGE = False` and `RUN_Q3V3 = False` (the default) the notebook
reuses the saved answers in `outputs/` and finishes in seconds. Set them to `True` to call the model again
(about 400 calls; roughly 45 minutes on a laptop, with Ollama running).

## 0. Check the environment
Prints which Python is running and checks that the packages import.

In [1]:
import sys
print(sys.executable)
import pandas, sklearn, ollama
print("imports OK")

/opt/miniconda3/bin/python
imports OK


## 1. Load my labels
50 items from the Bitext dataset, 10 with synthetic problems (`synthetic = 1`). How they were built: `prepare_data.py`.

In [2]:
import pandas as pd

labels = pd.read_excel("data/labels.xlsx")

for q in ["q1_human", "q2_human", "q3_human"]:
    labels[q] = labels[q].fillna("n/a").astype(str).str.strip().str.lower()
    print(q, labels[q].value_counts().to_dict())
print("synthetic ids:", labels.loc[labels["synthetic"] == 1, "id"].tolist())

q1_human {'yes': 26, 'n/a': 21, 'no': 3}
q2_human {'n/a': 30, 'no': 16, 'yes': 4}
q3_human {'no': 44, 'yes': 6}
synthetic ids: [1, 3, 5, 7, 21, 22, 26, 28, 31, 33]


## 2. The questions and the two prompt versions
v1 = bare question. v2 = rules + examples + evidence quote. Q2 gets the refund policy in both.

In [3]:
QUESTIONS = {
    "q1": "Does the customer's request need their order or account to be looked up, while the agent's reply does NOT ask for or mention an order number, account, or email?",
    "q2": "Does the agent's reply break the refund policy?",
    "q3": "Does the agent fail to acknowledge the customer's specific problem?",
}

RULES = {
    "q1": "- yes: the request needs a lookup and the reply does not ask for or mention any identifier\n- no: the reply asks for or mentions an order number, account or email\n- n/a: the request needs no lookup, e.g. a general question about policy",
    "q2": "- yes: the reply promises something the policy forbids: a guaranteed refund before checking, cash, faster than 5-10 business days, or a refund after 30 days\n- no: refunds are discussed and everything said fits the policy\n- n/a: refunds are not discussed at all",
    "q3": "- yes: the reply is generic only, e.g. 'I understand' or 'thanks for reaching out', and says nothing about this customer's specific problem\n- no: the reply names or restates the specific problem",
}

EXAMPLES = {
    "q1": "Customer: 'Where is my package?' Agent: 'Could you share your order number so I can check?' -> no\nCustomer: 'Where is my package?' Agent: 'Packages usually arrive soon, please wait.' -> yes",
    "q2": "Agent: 'Once we've checked your request, the refund goes back to your card within 5-10 business days.' -> no\nAgent: 'Don't worry, your refund is approved and will arrive today.' -> yes",
    "q3": "Customer: 'My blender arrived broken.' Agent: 'I'm sorry your blender arrived broken, let's fix that.' -> no\nCustomer: 'My blender arrived broken.' Agent: 'Thank you for contacting us. How can I help?' -> yes",
}

POLICY = open("prompts/policy.txt").read()

def build_prompt(q, version, instruction, response):
    context = POLICY + "\n\n" if q == "q2" else ""
    base = ("You are a quality assurance reviewer for customer support.\n\n"
            + context
            + "Customer message:\n" + instruction + "\n\n"
            + "Agent reply:\n" + response + "\n\n"
            + "Question: " + QUESTIONS[q] + "\n")
    if version == "v1":
        return base + "Answer yes, no, or n/a."
    return (base
            + "Judge only what is written in the agent reply.\n\n"
            + "Rules:\n" + RULES[q] + "\n\n"
            + "Examples:\n" + EXAMPLES[q] + "\n\n"
            + "First copy the exact sentence from the agent reply your answer is based on, or 'none'. Then answer yes, no, or n/a.")

## 3. Ask the model once (test)

In [4]:
import json, re, ollama

MODEL = "qwen2.5:7b-instruct"

SCHEMA = {
    "type": "object",
    "properties": {
        "evidence": {"type": "string"},
        "answer": {"type": "string", "enum": ["yes", "no", "n/a"]},
    },
    "required": ["evidence", "answer"],
}

def ask(prompt):
    r = ollama.chat(model=MODEL,
                    messages=[{"role": "user", "content": prompt}],
                    format=SCHEMA,
                    options={"temperature": 0})
    return r["message"]["content"]

def parse(raw):
    try:
        data = json.loads(re.search(r"\{.*\}", raw, re.S).group(0))
        answer = data["answer"].strip().lower()
        return answer if answer in {"yes", "no", "n/a"} else "invalid"
    except Exception:
        return "invalid"

# test on one item: id 7, one of your synthetic refund violations
item = labels[labels["id"] == 7].iloc[0]
prompt = build_prompt("q2", "v2", item["instruction"], item["response"])
raw = ask(prompt)
print(raw)
print("model says:", parse(raw), "| you said:", item["q2_human"])

{
"evidence": "Even though it's been 3 months, I'll make an exception and refund you.",
"answer": "yes"
}
model says: yes | you said: yes


## 4. Run the judge: 3 questions × 2 versions × 50 items

In [5]:
RUN_JUDGE = False   # True = call the model again (300 calls); False = reuse outputs/judge_answers.csv

if RUN_JUDGE:
    rows = []
    for _, item in labels.iterrows():
        for q in ["q1", "q2", "q3"]:
            for version in ["v1", "v2"]:
                raw = ask(build_prompt(q, version, item["instruction"], item["response"]))
                rows.append({"id": item["id"], "question": q, "version": version,
                             "raw": raw, "judge": parse(raw)})
        print("done item", item["id"])
    answers = pd.DataFrame(rows)
    answers.to_csv("outputs/judge_answers.csv", index=False)
else:
    answers = pd.read_csv("outputs/judge_answers.csv", keep_default_na=False)

print(answers.groupby(["question", "version"])["judge"].value_counts().unstack(fill_value=0))

judge             n/a  no  yes
question version              
q1       v1        27  20    3
         v2         3  43    4
q2       v1        36   5    9
         v2        24  22    4
q3       v1        11  11   28
         v2         0  24   26


## 5. Q1 v3: code decides whether a lookup is needed, the model only checks for an identifier
Caveat: my Q1 labels follow the same request-type rule, so the "needs a lookup" half agrees with me by design. The real test is the identifier half. Id 23 is a complaint I labelled n/a.

In [6]:
ASKS_Q = ("Does the agent's reply ask the customer for, or mention, a specific identifier: "
          "an order number, an account, or an email address? "
          "A general request for 'details', 'documentation' or 'contact information' does not count.")

SCHEMA_YN = {"type": "object",
             "properties": {"evidence": {"type": "string"},
                            "answer": {"type": "string", "enum": ["yes", "no"]}},
             "required": ["evidence", "answer"]}

def ask_yn(question, instruction, response):
    prompt = ("You are a quality assurance reviewer for customer support.\n\n"
              "Customer message:\n" + instruction + "\n\n"
              "Agent reply:\n" + response + "\n\n"
              "Question: " + question + "\n"
              "First copy the exact sentence your answer is based on, or 'none'. Then answer yes or no.")
    r = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}],
                    format=SCHEMA_YN, options={"temperature": 0})
    return parse(r["message"]["content"])

LOOKUP_INTENTS = {"get_refund", "track_refund", "track_order", "change_order", "complaint"}

if RUN_JUDGE:
    rows = []
    for _, item in labels.iterrows():
        if item["intent"] not in LOOKUP_INTENTS:
            asks, final = "", "n/a"          # code decides: no lookup needed
        else:
            asks = ask_yn(ASKS_Q, item["instruction"], item["response"])
            final = {"no": "yes", "yes": "no"}.get(asks, "invalid")
        rows.append({"id": item["id"], "question": "q1", "version": "v3",
                     "asks": asks, "judge": final, "raw": ""})
        print("done item", item["id"])
    v3 = pd.DataFrame(rows)
    v3.to_csv("outputs/judge_answers_q1_v3.csv", index=False)
else:
    v3 = pd.read_csv("outputs/judge_answers_q1_v3.csv", keep_default_na=False)

print(v3["judge"].value_counts())

judge
yes    28
n/a    20
no      2
Name: count, dtype: int64


## 5b. Q3 v3: ask the positive question, invert the answer in code
Tests whether the negative wording ("does the agent *fail* to…") caused Q3's false alarms.

In [7]:
ACK_Q = ("Does the agent's reply name or restate the customer's specific request or problem "
         "(for example 'you would like to check your order status')? "
         "Generic phrases such as 'Thanks for reaching out' or 'How can I help?' do not count.")

def ask_yn_raw(question, instruction, response):
    prompt = ("You are a quality assurance reviewer for customer support.\n\n"
              "Customer message:\n" + instruction + "\n\n"
              "Agent reply:\n" + response + "\n\n"
              "Question: " + question + "\n"
              "First copy the exact sentence your answer is based on, or 'none'. Then answer yes or no.")
    r = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}],
                    format=SCHEMA_YN, options={"temperature": 0})
    return r["message"]["content"]

RUN_Q3V3 = False   # True = call the model again (50 calls); False = reuse the saved answers

if RUN_Q3V3:
    rows = []
    for _, item in labels.iterrows():
        raw = ask_yn_raw(ACK_Q, item["instruction"], item["response"])
        ack = parse(raw)
        final = {"yes": "no", "no": "yes"}.get(ack, "invalid")   # no acknowledgement = problem
        rows.append({"id": item["id"], "question": "q3", "version": "v3",
                     "ack": ack, "judge": final, "raw": raw})
        print("done item", item["id"])
    v3_q3 = pd.DataFrame(rows)
    v3_q3.to_csv("outputs/judge_answers_q3_v3.csv", index=False)
else:
    v3_q3 = pd.read_csv("outputs/judge_answers_q3_v3.csv", keep_default_na=False)

print(v3_q3["judge"].value_counts())

judge
yes    27
no     23
Name: count, dtype: int64


## 6. Score every question and version
`yes` = problem found = positive. Invalid answers always count as wrong.

In [8]:
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             matthews_corrcoef, cohen_kappa_score)

long = labels.melt(id_vars=["id", "synthetic"],
                   value_vars=["q1_human", "q2_human", "q3_human"],
                   var_name="question", value_name="human")
long["question"] = long["question"].str.replace("_human", "")

all_answers = pd.concat([answers,
                         v3[["id", "question", "version", "raw", "judge"]],
                         v3_q3[["id", "question", "version", "raw", "judge"]]])
data = all_answers.merge(long, on=["id", "question"])

def binarize(g):
    h = (g["human"] == "yes").astype(int).to_numpy()
    j = (g["judge"] == "yes").astype(int).to_numpy().copy()
    bad = (g["judge"] == "invalid").to_numpy()
    j[bad] = 1 - h[bad]          # an invalid answer always counts as wrong
    return h, j

def score(data):
    rows = []
    for (q, v), g in data.groupby(["question", "version"]):
        h, j = binarize(g)
        tn, fp, fn, tp = confusion_matrix(h, j, labels=[0, 1]).ravel()
        rows.append({"question": q, "version": v,
                     "real_yes": int(h.sum()), "flagged": int(j.sum()),
                     "hits": tp, "false_alarms": fp, "missed": fn,
                     "invalid": int((g["judge"] == "invalid").sum()),
                     "precision": precision_score(h, j, zero_division=0),
                     "recall": recall_score(h, j, zero_division=0),
                     "mcc": matthews_corrcoef(h, j),
                     "kappa_binary": cohen_kappa_score(h, j),
                     "kappa_3_labels": cohen_kappa_score(g["human"], g["judge"])})
    return pd.DataFrame(rows).round(2)

results = score(data)
results.to_csv("outputs/results.csv", index=False)
results

,question,version,real_yes,flagged,hits,false_alarms,missed,invalid,precision,recall,mcc,kappa_binary,kappa_3_labels
0,q1,v1,26,3,3,0,23,0,1.00,0.12,0.24,0.11,0.19
1,q1,v2,26,4,2,2,24,0,0.50,0.08,-0.01,-0.01,0.05
2,q1,v3,26,28,26,2,0,0,0.93,1.00,0.92,0.92,0.93
3,q2,v1,4,9,4,5,0,0,0.44,1.00,0.63,0.57,0.46
4,q2,v2,4,4,4,0,0,0,1.00,1.00,1.00,1.00,0.72
5,q3,v1,6,28,6,22,0,0,0.21,1.00,0.33,0.19,0.11
6,q3,v2,6,26,6,20,0,0,0.23,1.00,0.35,0.22,0.22
7,q3,v3,6,27,6,21,0,0,0.22,1.00,0.34,0.21,0.21


## 7. Look at disagreements
Change `Q` and `V` to inspect any question and version.

In [9]:
pd.set_option("display.max_colwidth", 300)
Q, V = "q1", "v2"
items = labels[["id", "instruction", "response"]]
wrong = data[(data.question == Q) & (data.version == V)
             & (data.human != data.judge)].merge(items, on="id")
print(len(wrong), "disagreements")
wrong[["id", "human", "judge", "instruction", "raw"]]

42 disagreements


,id,human,judge,instruction,raw
0,1,n/a,no,I need to check in which cases can I ask for a reimbursement,"{\n""evidence"": ""If you have any concerns regarding a specific order or need assistance with a potential reimbursement request, please provide me with the necessary details, such as the order number, and I'll be more than happy to assist you further."",\n""answer"": ""no""\n}"
1,3,yes,no,I don't know how I can get a restitution of my money,"{\n""evidence"": ""none"",\n""answer"": ""no""\n}"
2,4,n/a,no,I have to see in what cases can I request a refund,"{\n""evidence"": ""Please keep in mind that each company or vendor may have specific refund policies, so it's advisable to review their terms and conditions or contact their customer support for more detailed information. If you have any specific concerns or questions related to a particular order ..."
3,5,yes,no,how can I get refunds of my money?,"{\n""evidence"": ""It's important to have your proof of purchase or any relevant documentation ready when contacting customer support."",\n""answer"": ""no""\n}"
4,6,yes,no,I expect a rebate of {{Refund Amount}} dollars,"{\n""evidence"": ""While I do that, could you please confirm whether you have received any communication regarding the rebate?"",\n""answer"": ""no""\n}"
5,7,yes,no,I am waiting for a compensation of {{Refund Amount}} doolars,"{\n""evidence"": ""While I do that, could you please provide me with any relevant details or documentation that may assist us in expediting the process?"",\n""answer"": ""no""\n}"
6,8,n/a,no,i try to see ur reimbursement policy,"{\n""evidence"": ""I recommend reviewing the terms and conditions provided during your purchase or reaching out to our customer support team for further assistance."",\n""answer"": ""no""\n}"
7,9,n/a,yes,help to check in which cases can I ask to be refunded,"{\n""evidence"": ""If you have any specific order-related concerns, please provide me with the necessary details such as the order number, and I will be happy to assist you further."",\n""answer"": ""yes""\n}"
8,10,yes,no,I have got to know if there are any news on my compensation,"{\n""evidence"": ""May I ask if you have any specific concerns or questions regarding your compensation?"",\n""answer"": ""no""\n}"
9,11,n/a,no,can you help me seeing in what cases can I request a refund?,"{\n""evidence"": ""If you have any specific order or concern, please provide me with the relevant details, such as the {{Order Number}} or other pertinent information, and I will be happy to assist you further."",\n""answer"": ""no""\n}"


## 8. Confidence intervals (bootstrap)

In [10]:
import numpy as np
rng = np.random.default_rng(42)

def bootstrap_ci(h, j, metric, n_boot=2000):
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(h), len(h))   # draw items with replacement
        if h[idx].sum() == 0:
            continue                             # skip draws with no positives
        vals.append(metric(h[idx], j[idx]))
    low, high = np.percentile(vals, [2.5, 97.5])
    return round(low, 2), round(high, 2)

for (q, v), g in data.groupby(["question", "version"]):
    h, j = binarize(g)
    print(q, v,
          "| recall CI:", bootstrap_ci(h, j, lambda a, b: recall_score(a, b, zero_division=0)),
          "| MCC CI:", bootstrap_ci(h, j, matthews_corrcoef))

q1 v1 | recall CI: (np.float64(0.0), np.float64(0.26)) | MCC CI: (np.float64(0.0), np.float64(0.38))
q1 v2 | recall CI: (np.float64(0.0), np.float64(0.2)) | MCC CI: (np.float64(-0.28), np.float64(0.25))
q1 v3 | recall CI: (np.float64(1.0), np.float64(1.0)) | MCC CI: (np.float64(0.81), np.float64(1.0))
q2 v1 | recall CI: (np.float64(1.0), np.float64(1.0)) | MCC CI: (np.float64(0.35), np.float64(0.88))
q2 v2 | recall CI: (np.float64(1.0), np.float64(1.0)) | MCC CI: (np.float64(1.0), np.float64(1.0))
q3 v1 | recall CI: (np.float64(1.0), np.float64(1.0)) | MCC CI: (np.float64(0.18), np.float64(0.46))
q3 v2 | recall CI: (np.float64(1.0), np.float64(1.0)) | MCC CI: (np.float64(0.19), np.float64(0.52))
q3 v3 | recall CI: (np.float64(1.0), np.float64(1.0)) | MCC CI: (np.float64(0.2), np.float64(0.49))


## 9. Is the newer version really better? (paired bootstrap)

In [11]:
def paired_bootstrap(q, old, new, metric=matthews_corrcoef, n_boot=2000):
    g_old = data[(data.question == q) & (data.version == old)].sort_values("id")
    g_new = data[(data.question == q) & (data.version == new)].sort_values("id")
    h_old, j_old = binarize(g_old)
    h_new, j_new = binarize(g_new)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(h_old), len(h_old))      # SAME items for both versions
        if h_old[idx].sum() == 0:
            continue                                         # skip draws with no positives
        diffs.append(metric(h_new[idx], j_new[idx]) - metric(h_old[idx], j_old[idx]))
    diffs = np.array(diffs)
    return {"mean_diff": round(float(diffs.mean()), 2),
            "CI": [round(float(x), 2) for x in np.percentile(diffs, [2.5, 97.5])],
            "share_new_better": round(float((diffs > 0).mean()), 2)}

for q, old, new in [("q1", "v1", "v2"), ("q1", "v2", "v3"),
                    ("q2", "v1", "v2"), ("q3", "v1", "v2"), ("q3", "v2", "v3")]:
    print(q, new, "vs", old, paired_bootstrap(q, old, new))

q1 v2 vs v1 {'mean_diff': -0.24, 'CI': [-0.56, 0.09], 'share_new_better': 0.07}
q1 v3 vs v2 {'mean_diff': 0.94, 'CI': [0.65, 1.22], 'share_new_better': 1.0}
q2 v2 vs v1 {'mean_diff': 0.38, 'CI': [0.13, 0.65], 'share_new_better': 0.99}
q3 v2 vs v1 {'mean_diff': 0.03, 'CI': [-0.07, 0.14], 'share_new_better': 0.64}
q3 v3 vs v2 {'mean_diff': -0.02, 'CI': [-0.15, 0.12], 'share_new_better': 0.36}


## 10. A simple baseline for Q3: word overlap
Flag a reply if it shares no content words with the customer's message (placeholders like `{{Order Number}}` and
very common words ignored, words cut to their first 5 letters). No LLM involved.
Caveat: I wrote this after seeing the data, and the 6 Q3 positives are fully generic replies, so they are easy.

In [12]:
STOP = set(("i me my you your we our us to a an the of in on for with and or is are be do does can could "
            "how what want need help have has get got some any there this that it if at by from please "
            "would like know new ur u").split())

def content_words(text):
    text = re.sub(r"\{\{.*?\}\}", " ", text.lower())          # drop {{placeholders}}
    return {w[:5] for w in re.findall(r"[a-z]+", text) if w not in STOP and len(w) > 2}

baseline = pd.DataFrame({
    "id": labels["id"], "question": "q3", "version": "baseline", "raw": "",
    "judge": ["yes" if not (content_words(i) & content_words(r)) else "no"
              for i, r in zip(labels["instruction"], labels["response"])],
})
score(baseline.merge(long, on=["id", "question"]))

,question,version,real_yes,flagged,hits,false_alarms,missed,invalid,precision,recall,mcc,kappa_binary,kappa_3_labels
0,q3,baseline,6,7,6,1,0,0,0.86,1.0,0.92,0.91,0.91


## 11. Error taxonomy
Every disagreement (Q1 v2 and v3, Q2 v1 and v2, Q3 v2), tagged by hand with one cause in `outputs/error_taxonomy.xlsx`.

In [13]:
tax = pd.read_excel("outputs/error_taxonomy.xlsx", sheet_name="errors")
pd.crosstab(tax["cause"], tax["question"], margins=True, margins_name="total")

question,q1,q2,q3,total
cause,,,,
model error,1,1,20,22
question ambiguity,27,12,0,39
total,28,13,20,61
